# Etapa 1 — Transformação de Dedos (Bronze → Silver)

**Objetivo:** transformar a camada **bronze** (CSVs brutos do BNDES) em uma
camada **silver** limpa, tipada, padronizada e enriquecida, pronta para
alimentar as etapas seguintes (EDA, engenharia, regressão, inferência,
multivariada, modelo).

**O que fazemos aqui:**
1. Converter CSVs da bronze em Parquet (preserva tipos, compacta, acelera).
2. Aplicar **regras de negócio** para valores nulos (documentadas uma a uma).
3. Padronizar **tipos de dados** (datas, numéricos).
4. Padronizar **texto** (uppercase, sem acentos, sem espaços duplicados).
5. **Feature engineering** (datas, prazos, taxas, flags, ordinais).
6. **Alinhar schema** com o dicionário silver (renomear colunas).
7. **Validar** Silver × Raw (contagem, nulos, somas de valores financeiros).
8. Salvar em `data/silver/`.


## Regras de negócio aplicadas nesta etapa

| Coluna | Regra | Justificativa |
|---|---|---|
| `tipo_excepcionalidade` | Nulo → operação seguiu o fluxo **padrão** → vira `tem_excepcionalidade` (0/1) e a coluna original é removida. | A ausência de exceção é o caso padrão; tratar nulo como "sem exceção" evita descartar linhas. |
| `cnpj_instituicao_financeira_credenciada` | Nulo → `"00000000000000.0"` (placeholder de 14 dígitos). | Nulo significa operação **direta com o BNDES**, sem instituição intermediária. |
| `nome_instituicao_financeira_credenciada` | Nulo → `"OPERAÇÃO DIRETA"`. | Idem. |
| `id_municipio` | Nulo → `"NÃO INFORMADO"`. | Município desconhecido não deve ser inferido; sinalizar explicitamente. |
| `cnpj_cliente` | Nulo → `"000000000000.0"` (placeholder). | Cliente não identificado; placeholder permite manter a linha. |
| `situacao_contrato` | Nulo → `"OUTROS"`. | Preserva a linha em vez de descartar por falta de situação. |
| `tipo_fonte_recursos` | Nulo → `"OUTROS"`. | Idem. |

#### Leitura & transformação

In [8]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# CORREÇÃO AQUI: Subindo dois níveis com '../../'
BRONZE_DIR = Path("../../data/bronze")
SILVER_DIR = Path("../../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Nome canônico do arquivo silver
NOME_ARQUIVO_SILVER = "operacoes_bndes_silver.parquet"
ARQUIVO_BRONZE_CSV  = "br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv"

# Tipos esperados na silver (para validação de schema)
# Tipos esperados na silver (para validação de schema)
ESQUEMA_SILVER = {
    "data_contratacao": "datetime64[ns]",
    "sigla_uf": "object", "id_municipio": "object", "id_contrato": "object",
    "cnpj_cliente": "object", "nome_cliente": "object",
    "porte_cliente": "object", "natureza_cliente": "object",
    "descricao_projeto": "object",
    "fonte_recurso": "object", "custo_financeiro": "object",
    "modalidade_apoio": "object", "forma_apoio": "object",
    "produto": "object", "instrumento_financeiro": "object",
    "tipo_garantia": "object",
    "indicador_inovacao": "int64",
    "area_operacional": "object",
    "setor_cnae": "object", "subsetor_cnae_agrupado": "object",
    "descricao_cnae": "object",
    "setor_bndes": "object", "subsetor_bndes": "object",
    "nome_instituicao_financeira_credenciada": "object",
    "cnpj_instituicao_financeira_credenciada": "object",
    "valor_operacao": "float64",
    "valor_desembolsado": "float64",
    "taxa_juros": "float64",
    "prazo_carencia": "float64",
    "prazo_amortizacao": "float64",
    "situacao_operacao": "object",
    # derivadas
    "ano_contratacao": "float64", "mes_contratacao": "float64",
    "prazo_total_meses": "float64", "qtd_palavras_descricao": "int64",
    "flag_apoio_direto": "int64", "flag_reembolsavel": "int64",
    "flag_cliente_publico": "int64", "porte_cliente_ordinal": "float64",
    "grupo_taxa_juros_ordinal": "int64", "tem_excepcionalidade": "int64",
}

In [9]:
# =============================================================
# 2. CONVERSÃO BRONZE (CSV) → SILVER (PARQUET BRUTO)
# =============================================================
def converter_csv_para_parquet(dir_bronze: Path, dir_silver: Path) -> None:
    """Converte todos os CSVs da bronze em Parquet, preservando nomes."""
    arquivos = sorted([f for f in os.listdir(dir_bronze)
                       if f.lower().endswith(".csv")])
    if not arquivos:
        print("[bronze→silver] nenhum CSV encontrado.")
        return
    for arq in arquivos:
        caminho_in  = dir_bronze / arq
        caminho_out = dir_silver / f"{Path(arq).stem}.parquet"
        df = pd.read_csv(caminho_in)
        df.to_parquet(caminho_out, index=False)
        print(f"  {arq} → {caminho_out.name}")

converter_csv_para_parquet(BRONZE_DIR, SILVER_DIR)

  br_bd_diretorios_brasil_cnae_2.csv → br_bd_diretorios_brasil_cnae_2.parquet
  br_bd_diretorios_brasil_municipio.csv → br_bd_diretorios_brasil_municipio.parquet
  br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv → br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet


## 2. Carregamento da base de operações

Esta é a **única** tabela que usamos nas etapas seguintes: o histórico de
operações contratadas não automáticas do BNDES.

In [10]:
# =============================================================
# 3. CARREGAMENTO E INSPEÇÃO INICIAL
# =============================================================
operacoes = pd.read_parquet(SILVER_DIR / "br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet")
print(f"Shape original: {operacoes.shape[0]:,} × {operacoes.shape[1]}")
print("\nTipos originais:")
print(operacoes.dtypes.value_counts())
operacoes.head(2)

Shape original: 23,483 × 39

Tipos originais:
str        25
float64     9
int64       5
Name: count, dtype: int64


,razao_social_cliente,cnpj_cliente,descricao_projeto,sigla_uf,nome_municipio,id_municipio,id_contrato,data_contratacao,valor_contratado,valor_desembolsado,...,cnpj_instituicao_financeira_credenciada,tipo_garantia,tipo_excepcionalidade,situacao_contrato,secao_cnae,divisao_cnae,grupo_cnae,classe_cnae,subclasse_cnae,data_apuracao
0,COPACOL-COOPERATIVA AGROINDUSTRIAL CONSOLATA,7.609373e+13,INVESTIMENTOS EM AMPLIACAO DA CAPACIDADE DE RE...,PR,SEM MUNICÍPIO,NaN,9203361,2009-06-30,35900000.0,35900000.0,...,9.281656e+13,DEFINIDA PELO AGENTE FINANCEIRO,NaN,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04
1,LAR COOPERATIVA AGROINDUSTRIAL,7.775229e+13,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,PR,MATELANDIA,4115606.0,9203571,2009-06-29,66998000.0,66998000.0,...,NaN,REAL,NaN,LIQUIDADO,C,10,101.0,10121.0,1012101.0,2026-06-04


In [11]:
# =============================================================
# 4. HELPER DE VERIFICAÇÃO DE NULOS (reutilizado adiante)
# =============================================================
def verificar_valores_nulos(df: pd.DataFrame) -> pd.DataFrame | None:
    """Retorna tabela de colunas com nulos, ou None se não houver."""
    total = df.isnull().sum().sort_values(ascending=False)
    total = total[total > 0]
    if total.empty:
        return None
    return pd.DataFrame({
        "Total Nulls": total,
        "%": ((total / len(df)) * 100).round(2),
    })

print("Distribuição de nulos ANTES do tratamento:")
display(verificar_valores_nulos(operacoes))

Distribuição de nulos ANTES do tratamento:


,Total Nulls,%
tipo_excepcionalidade,23265,99.07
cnpj_instituicao_financeira_credenciada,19412,82.66
nome_instituicao_financeira_credenciada,19412,82.66
id_municipio,7721,32.88
subclasse_cnae,5507,23.45
tipo_fonte_recursos,836,3.56
classe_cnae,497,2.12
situacao_contrato,268,1.14
grupo_cnae,255,1.09
cnpj_cliente,9,0.04


## 3. Tratamento de nulos — por regra de negócio

**Nenhum nulo é tratado por "achismo".** Cada preenchimento tem uma regra de
negócio documentada (tabela no início do notebook).

In [14]:
# =============================================================
# 5. REGRA 1 — tipo_excepcionalidade → tem_excepcionalidade (0/1)
# =============================================================
# nulo = operação no fluxo padrão (sem exceção) → 0
print("Valores originais de tipo_excepcionalidade:")
print(operacoes["tipo_excepcionalidade"].value_counts(dropna=False).head())

operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

assert operacoes["tem_excepcionalidade"].isin([0, 1]).all()
print("\nDistribuição após transformação:")
print(operacoes["tem_excepcionalidade"].value_counts())

Valores originais de tipo_excepcionalidade:
tipo_excepcionalidade
NaN                                                           23265
CONDIÇÕES DE CRÉDITO                                             81
CONDIÇÕES FINANCEIRAS E OPERACIONAIS                             68
COBRANÇA DE COMISSÕES                                            47
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /CONDIÇÕES DE CRÉDITO       17
Name: count, dtype: int64

Distribuição após transformação:
tem_excepcionalidade
0    23265
1      218
Name: count, dtype: int64


In [15]:
# =============================================================
# 6. REGRA 2 — instituição financeira nula = operação DIRETA
# =============================================================
PLACEHOLDER_CNPJ = "00000000000000.0"   # 14 dígitos + .0 (formato CNPJ)

operacoes["cnpj_instituicao_financeira_credenciada"] = (
    operacoes["cnpj_instituicao_financeira_credenciada"].fillna(PLACEHOLDER_CNPJ)
)
operacoes["nome_instituicao_financeira_credenciada"] = (
    operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
)

print("Nulos restantes em instituição financeira:")
print(operacoes[[
    "cnpj_instituicao_financeira_credenciada",
    "nome_instituicao_financeira_credenciada",
]].isnull().sum())

Nulos restantes em instituição financeira:
cnpj_instituicao_financeira_credenciada    0
nome_instituicao_financeira_credenciada    0
dtype: int64


In [16]:
# =============================================================
# 7. REGRAS 3–6 — demais nulos
# =============================================================
# id_municipio → "NÃO INFORMADO"
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")

# cnpj_cliente → placeholder
PLACEHOLDER_CNPJ_CLIENTE = "000000000000.0"
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna(PLACEHOLDER_CNPJ_CLIENTE)

# situacao_contrato → "OUTROS"
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")

# tipo_fonte_recursos → "OUTROS"
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

print("Nulos remanescentes após todas as regras:")
print(verificar_valores_nulos(operacoes))

Nulos remanescentes após todas as regras:
                Total Nulls      %
subclasse_cnae         5507  23.45
classe_cnae             497   2.12
grupo_cnae              255   1.09


## 4. Remoção de CNAE granular

As colunas `secao_cnae`, `divisao_cnae`, `grupo_cnae`, `classe_cnae` e
`subclasse_cnae` formam uma **hierarquia redundante** com
`setor_bndes`/`subsetor_bndes`, que já trazem a granularidade usada pelo
BNDES nas suas análises. Mantê-las infla a cardinalidade e não agrega sinal
para as etapas seguintes.

**Decisão documentada:** removemos essas colunas e mantemos
`setor_bndes`, `subsetor_bndes` e `descricao_cnae`.

In [17]:
# =============================================================
# 8. DROP CNAE GRANULAR
# =============================================================
COLUNAS_CNAE_REDUNDANTES = [
    "secao_cnae", "divisao_cnae", "grupo_cnae",
    "classe_cnae", "subclasse_cnae",
]
operacoes = operacoes.drop(
    columns=[c for c in COLUNAS_CNAE_REDUNDANTES if c in operacoes.columns]
)
print(f"Após drop CNAE: {operacoes.shape[1]} colunas")

Após drop CNAE: 34 colunas


In [18]:
# =============================================================
# 9. PADRONIZAÇÃO DE TIPOS
# =============================================================
# Datas
cols_data = [c for c in operacoes.columns if "data" in c.lower()]
operacoes[cols_data] = operacoes[cols_data].apply(
    lambda s: pd.to_datetime(s, errors="coerce")
)
print(f"Datas convertidas: {cols_data}")

# Numéricos
cols_num = ["valor_contratado", "valor_desembolsado",
            "taxa_juros", "prazo_carencia", "prazo_amortizacao"]
for col in cols_num:
    if col in operacoes.columns:
        operacoes[col] = pd.to_numeric(operacoes[col], errors="coerce")

# indicador_inovacao
if "indicador_inovacao" in operacoes.columns:
    operacoes["indicador_inovacao"] = pd.to_numeric(
        operacoes["indicador_inovacao"], errors="coerce"
    ).fillna(0).astype(int)

print("\nTipos após padronização:")
print(operacoes[cols_num].dtypes)

Datas convertidas: ['data_contratacao', 'data_apuracao']

Tipos após padronização:
valor_contratado      float64
valor_desembolsado    float64
taxa_juros            float64
prazo_carencia          int64
prazo_amortizacao       int64
dtype: object


## 6. Padronização de texto

Aplicamos **uppercase**, **remoção de acentos** e **colapso de espaços** em
todas as colunas de texto. Objetivo:

- Permitir comparações exatas (`==`) e regex confiáveis nas etapas seguintes.
- Eliminar variantes de grafia ("Pública" × "PUBLICA" × "Publica").
- Manter a semântica original sem perda (apenas forma).

**Não confundir com "normalização de sinônimos"** — isso fica para a etapa 2,
quando agrupamos categorias raras em `"OUTROS"`.

In [19]:
# =============================================================
# 10. PADRONIZAÇÃO DE TEXTO
# =============================================================
def normalizar_texto(valor) -> str | None:
    """Uppercase + sem acentos + sem espaços duplicados."""
    if pd.isna(valor):
        return valor
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r"\s+", " ", texto)
    return texto


def padronizar_texto(df: pd.DataFrame,
                     cols: list[str] | None = None) -> pd.DataFrame:
    """Aplica `normalizar_texto` em todas as colunas de texto (ou em `cols`)."""
    out = df.copy()
    if cols is None:
        cols = out.select_dtypes(include=["object", "string"]).columns
    for col in cols:
        out[col] = out[col].map(normalizar_texto)
    return out


operacoes = padronizar_texto(operacoes)
print("Texto padronizado. Amostra de valores únicos:")
for col in ["porte_cliente", "modalidade_apoio", "forma_apoio", "natureza_cliente"]:
    if col in operacoes.columns:
        print(f"  {col}: {operacoes[col].unique()[:5]}")

Texto padronizado. Amostra de valores únicos:
  porte_cliente: <ArrowStringArray>
['GRANDE', 'SEM PORTE', 'MEDIA', 'PEQUENA', 'MICRO']
Length: 5, dtype: str
  modalidade_apoio: <ArrowStringArray>
['REEMBOLSAVEL', 'NAO REEMBOLSAVEL']
Length: 2, dtype: str
  forma_apoio: <ArrowStringArray>
['INDIRETA', 'DIRETA']
Length: 2, dtype: str
  natureza_cliente: <ArrowStringArray>
[                                         'PRIVADA',
 'ADMINISTRACAO PUBLICA DIRETA - GOVERNO MUNICIPAL',
  'ADMINISTRACAO PUBLICA DIRETA - GOVERNO ESTADUAL',
                                 'PUBLICA INDIRETA',
   'ADMINISTRACAO PUBLICA DIRETA - GOVERNO FEDERAL']
Length: 5, dtype: str


## 7. Feature engineering

Criamos features derivadas de negócio na silver:

| Feature | Derivação | Por que |
|---|---|---|
| `ano_contratacao`, `mes_contratacao` | `data_contratacao` | Sazonalidade e validação temporal |
| `prazo_total_meses` | `prazo_carencia + prazo_amortizacao` | Visão consolidada do prazo |
| `grupo_taxa_juros` + `_ordinal` | bins de `taxa_juros` | Categoriza e ordena |
| `flag_apoio_direto` | `forma_apoio == "DIRETA"` | Direto × indireto |
| `flag_reembolsavel` | `modalidade_apoio == "REEMBOLSAVEL"` | Reembolsável × não |
| `flag_cliente_publico` | `natureza_cliente` contém "PUBLICA/PUBLICO" | Cliente público × privado |
| `qtd_palavras_descricao` | `descricao_projeto.split()` | Complexidade do projeto |
| `porte_cliente_ordinal` | mapa ordinal de `porte_cliente` | Captura ordem natural |

In [20]:
# =============================================================
# 11. FEATURE ENGINEERING
# =============================================================
def aplicar_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica as transformações da camada Silver no DataFrame de operações."""
    out = df.copy()

    # --- 1. Datas ------------------------------------------------------
    out["ano_contratacao"] = out["data_contratacao"].dt.year
    out["mes_contratacao"] = out["data_contratacao"].dt.month

    # --- 2. Prazo total -----------------------------------------------
    out["prazo_total_meses"] = (
        out["prazo_carencia"].fillna(0) + out["prazo_amortizacao"].fillna(0)
    )

    # --- 3. Grupo de taxa de juros (categorização) --------------------
    condicoes = [
        out["taxa_juros"] == 0,
        out["taxa_juros"] < 5,
        (out["taxa_juros"] >= 5) & (out["taxa_juros"] <= 10),
        out["taxa_juros"] > 10,
    ]
    categorias = ["SEM TAXA", "BAIXA", "MEDIA", "ALTA"]
    out["grupo_taxa_juros"] = np.select(condicoes, categorias, default="SEM TAXA")

    # --- 4. Grupo de taxa (ordinal) -----------------------------------
    mapa_taxa = {"SEM TAXA": 0, "BAIXA": 1, "MEDIA": 2, "ALTA": 3}
    out["grupo_taxa_juros_ordinal"] = out["grupo_taxa_juros"].map(mapa_taxa)

    # --- 5. Flags binárias -------------------------------------------
    out["flag_apoio_direto"] = (out["forma_apoio"] == "DIRETA").astype(int)
    out["flag_reembolsavel"] = (out["modalidade_apoio"] == "REEMBOLSAVEL").astype(int)
    out["flag_cliente_publico"] = (
        out["natureza_cliente"]
           .fillna("")
           .str.contains("PUBLICA|PUBLICO", regex=True)
           .astype(int)
    )

    # --- 6. Complexidade da descrição ---------------------------------
    out["qtd_palavras_descricao"] = (
        out["descricao_projeto"].fillna("").str.split().str.len().astype(int)
    )

    # --- 7. Porte ordinal ---------------------------------------------
    mapa_porte = {
        "SEM PORTE": 0, "MICRO": 1, "PEQUENA": 2,
        "MEDIA": 3, "GRANDE": 4,
    }
    out["porte_cliente_ordinal"] = out["porte_cliente"].map(mapa_porte)

    return out


operacoes = aplicar_feature_engineering(operacoes)
print(f"Shape após feature engineering: {operacoes.shape}")
display(operacoes.head(2))

Shape após feature engineering: (23483, 44)


,razao_social_cliente,cnpj_cliente,descricao_projeto,sigla_uf,nome_municipio,id_municipio,id_contrato,data_contratacao,valor_contratado,valor_desembolsado,...,ano_contratacao,mes_contratacao,prazo_total_meses,grupo_taxa_juros,grupo_taxa_juros_ordinal,flag_apoio_direto,flag_reembolsavel,flag_cliente_publico,qtd_palavras_descricao,porte_cliente_ordinal
0,COPACOL-COOPERATIVA AGROINDUSTRIAL CONSOLATA,76093731000190.0,INVESTIMENTOS EM AMPLIACAO DA CAPACIDADE DE RE...,PR,SEM MUNICIPIO,NAO INFORMADO,9203361,2009-06-30,35900000.0,35900000.0,...,2009,6,108,MEDIA,2,0,1,0,44,4
1,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,PR,MATELANDIA,4115606.0,9203571,2009-06-29,66998000.0,66998000.0,...,2009,6,108,MEDIA,2,1,1,0,24,4


In [21]:
# =============================================================
# 12. VERIFICAÇÃO DAS FLAGS CRIADAS
# =============================================================
verificacoes = [
    ("forma_apoio",          "flag_apoio_direto"),
    ("modalidade_apoio",     "flag_reembolsavel"),
    ("natureza_cliente",     "flag_cliente_publico"),
    ("porte_cliente",        "porte_cliente_ordinal"),
    ("grupo_taxa_juros",     "grupo_taxa_juros_ordinal"),
]

for origem, flag in verificacoes:
    print(f"\n=== {origem} × {flag} ===")
    print(operacoes.groupby(origem)[flag].first().sort_values())


=== forma_apoio × flag_apoio_direto ===
forma_apoio
INDIRETA    0
DIRETA      1
Name: flag_apoio_direto, dtype: int64

=== modalidade_apoio × flag_reembolsavel ===
modalidade_apoio
NAO REEMBOLSAVEL    0
REEMBOLSAVEL        1
Name: flag_reembolsavel, dtype: int64

=== natureza_cliente × flag_cliente_publico ===
natureza_cliente
PRIVADA                                             0
ADMINISTRACAO PUBLICA DIRETA - GOVERNO ESTADUAL     1
ADMINISTRACAO PUBLICA DIRETA - GOVERNO FEDERAL      1
ADMINISTRACAO PUBLICA DIRETA - GOVERNO MUNICIPAL    1
PUBLICA INDIRETA                                    1
Name: flag_cliente_publico, dtype: int64

=== porte_cliente × porte_cliente_ordinal ===
porte_cliente
SEM PORTE    0
MICRO        1
PEQUENA      2
MEDIA        3
GRANDE       4
Name: porte_cliente_ordinal, dtype: int64

=== grupo_taxa_juros × grupo_taxa_juros_ordinal ===
grupo_taxa_juros
SEM TAXA    0
BAIXA       1
MEDIA       2
ALTA        3
Name: grupo_taxa_juros_ordinal, dtype: int64


## 8. Alinhamento de schema ao dicionário silver

A bronze do BNDES usa nomes de coluna ligeiramente diferentes do dicionário
canônico adotado no projeto. **Padronizar aqui evita quebra em todas as etapas
seguintes** (que referenciam `valor_operacao`, `fonte_recurso`,
`situacao_operacao`).

| Nome na bronze | Nome canônico (dicionário silver) |
|---|---|
| `valor_contratado` | `valor_operacao` |
| `tipo_fonte_recursos` | `fonte_recurso` |
| `situacao_contrato` | `situacao_operacao` |

Além disso, registramos que `tipo_excepcionalidade` **não** será renomeada de
volta — a silver entrega `tem_excepcionalidade` (binário) por decisão de
negócio, e o dicionário deve ser atualizado para refletir isso.

In [22]:
# =============================================================
# 13. ALINHAMENTO DE SCHEMA — RENOMEAÇÕES
# =============================================================
MAPA_RENOMEACAO = {
    "valor_contratado":    "valor_operacao",
    "tipo_fonte_recursos": "fonte_recurso",
    "situacao_contrato":   "situacao_operacao",
}
operacoes = operacoes.rename(columns=MAPA_RENOMEACAO)

print("Colunas após alinhamento de schema:")
print(sorted(operacoes.columns.tolist()))

Colunas após alinhamento de schema:
['ano_contratacao', 'area_operacional_bndes', 'cnpj_cliente', 'cnpj_instituicao_financeira_credenciada', 'custo_financeiro', 'data_apuracao', 'data_contratacao', 'descricao_projeto', 'descricao_subclasse', 'flag_apoio_direto', 'flag_cliente_publico', 'flag_reembolsavel', 'fonte_recurso', 'forma_apoio', 'grupo_taxa_juros', 'grupo_taxa_juros_ordinal', 'id_contrato', 'id_municipio', 'indicador_inovacao', 'mes_contratacao', 'modalidade_apoio', 'natureza_cliente', 'nome_instituicao_financeira_credenciada', 'nome_municipio', 'porte_cliente', 'porte_cliente_ordinal', 'prazo_amortizacao', 'prazo_carencia', 'prazo_total_meses', 'produto', 'qtd_palavras_descricao', 'razao_social_cliente', 'setor_bndes', 'setor_cnae_bndes', 'sigla_uf', 'situacao_operacao', 'subsetor_agrupado_cnae_bndes', 'subsetor_bndes', 'taxa_juros', 'tem_excepcionalidade', 'tipo_garantia', 'tipo_instrumento_financeiro', 'valor_desembolsado', 'valor_operacao']


## 9. Validação Silver × Raw

Três checagens para garantir que nenhuma transformação corrompeu os dados:

1. **Contagem de linhas** — igual (nenhuma linha perdida).
2. **Somas de `valor_operacao` e `valor_desembolsado`** — iguais (somas
   dos valores financeiros preservadas).
3. **Contagem de nulos** por coluna — não aumenta por causa das
   transformações (exceto nos campos que a silver removeu por decisão).

In [23]:
# =============================================================
# 14. VALIDAÇÃO SILVER × RAW
# =============================================================
# Lê o CSV bruto como texto para preservar precisão
raw = pd.read_csv(BRONZE_DIR / ARQUIVO_BRONZE_CSV, dtype=str)

print(f"Contagem de linhas — raw: {len(raw):,} | silver: {len(operacoes):,}")
assert len(raw) == len(operacoes), \
    f"Contagem de linhas divergente: raw={len(raw)}, silver={len(operacoes)}"

# Somas de valores financeiros
cols_valor = [("valor_contratado", "valor_operacao"),
              ("valor_desembolsado", "valor_desembolsado")]

for col_raw, col_silver in cols_valor:
    soma_raw = pd.to_numeric(raw[col_raw], errors="coerce").sum()
    soma_silver = operacoes[col_silver].sum()
    diff_pct = abs(soma_raw - soma_silver) / soma_raw * 100 if soma_raw else 0
    print(f"\n{col_silver}:")
    print(f"  raw    = R$ {soma_raw:,.2f}")
    print(f"  silver = R$ {soma_silver:,.2f}")
    print(f"  diff   = {diff_pct:.4f}%")
    assert diff_pct < 0.01, (
        f"Soma divergente em {col_silver}: {diff_pct:.4f}%"
    )

print("\n✓ Validação Silver × Raw concluída com sucesso.")

Contagem de linhas — raw: 23,483 | silver: 23,483

valor_operacao:
  raw    = R$ 1,229,184,483,691.93
  silver = R$ 1,229,184,483,691.93
  diff   = 0.0000%

valor_desembolsado:
  raw    = R$ 952,046,450,761.74
  silver = R$ 952,046,450,761.74
  diff   = 0.0000%

✓ Validação Silver × Raw concluída com sucesso.


## 10. Validação de schema contra o dicionário

Garantimos que **as colunas obrigatórias existem** e que **não há colunas
inesperadas** que possam confundir as etapas seguintes. Falha cedo, alto e
claro — em vez de silenciosamente ignorar colunas.

In [24]:
# =============================================================
# 15. VALIDAÇÃO DE SCHEMA CONTRA O DICIONÁRIO SILVER
# =============================================================
colunas_esperadas = set(ESQUEMA_SILVER.keys())
colunas_atuais    = set(operacoes.columns)

faltantes = colunas_esperadas - colunas_atuais
extras    = colunas_atuais - colunas_esperadas

print(f"Colunas esperadas: {len(colunas_esperadas)}")
print(f"Colunas na silver: {len(colunas_atuais)}")

if faltantes:
    print(f"\n⚠ FALTANDO na silver ({len(faltantes)}):")
    print(sorted(faltantes))
else:
    print("\n✓ Nenhuma coluna obrigatória ausente.")

if extras:
    print(f"\n⚠ EXTRAS na silver ({len(extras)}):")
    print(sorted(extras))
    print("(Atualize o dicionário ou remova as colunas.)")
else:
    print("\n✓ Nenhuma coluna extra.")

# Checagem de tipo (apenas informativa)
print("\nTipos esperados vs atuais (amostra):")
for col, tipo_esp in list(ESQUEMA_SILVER.items())[:10]:
    if col in operacoes.columns:
        print(f"  {col:45s} esp={tipo_esp:20s} atual={str(operacoes[col].dtype)}")

Colunas esperadas: 41
Colunas na silver: 44

⚠ FALTANDO na silver (6):
['area_operacional', 'descricao_cnae', 'instrumento_financeiro', 'nome_cliente', 'setor_cnae', 'subsetor_cnae_agrupado']

⚠ EXTRAS na silver (9):
['area_operacional_bndes', 'data_apuracao', 'descricao_subclasse', 'grupo_taxa_juros', 'nome_municipio', 'razao_social_cliente', 'setor_cnae_bndes', 'subsetor_agrupado_cnae_bndes', 'tipo_instrumento_financeiro']
(Atualize o dicionário ou remova as colunas.)

Tipos esperados vs atuais (amostra):
  data_contratacao                              esp=datetime64[ns]       atual=datetime64[us]
  sigla_uf                                      esp=object               atual=str
  id_municipio                                  esp=object               atual=str
  id_contrato                                   esp=object               atual=int64
  cnpj_cliente                                  esp=object               atual=str
  porte_cliente                                 esp=object 

## 11. Salvamento da camada silver

Salvamos a silver canônica em **Parquet**, com nome **estável** e documentado.
As etapas seguintes sempre leem deste arquivo — sem "adivinhação" de nome.

In [25]:
# =============================================================
# 16. SALVAMENTO DA SILVER
# =============================================================
caminho_silver = SILVER_DIR / NOME_ARQUIVO_SILVER
operacoes.to_parquet(caminho_silver, index=False)
print(f"Silver salva em: {caminho_silver}")
print(f"Shape final: {operacoes.shape[0]:,} × {operacoes.shape[1]}")

# Relatório de tipos final
print("\nDistribuição de tipos na silver:")
print(operacoes.dtypes.value_counts())

Silver salva em: ..\..\data\silver\operacoes_bndes_silver.parquet
Shape final: 23,483 × 44

Distribuição de tipos na silver:
str               25
int64             12
float64            3
datetime64[us]     2
int32              2
Name: count, dtype: int64
